# Basic VaR Calculation Example

This notebook demonstrates how to calculate Value at Risk (VaR) using different methodologies.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
from market_risk_hub.data.market_data import MarketDataFetcher
from market_risk_hub.risk_engines.var import VaRCalculator
from market_risk_hub.utils.visualization import RiskVisualizer

## 1. Fetch Market Data

In [ ]:
# Initialize data fetcher
fetcher = MarketDataFetcher()

# Define portfolio
tickers = ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'GLD']

# Fetch data
data = fetcher.get_market_data(tickers, period='2y')
prices = data['prices']
returns = data['returns']

print(f"Data shape: {returns.shape}")
print(f"Date range: {returns.index[0]} to {returns.index[-1]}")
returns.head()

## 2. Define Portfolio Weights

In [ ]:
# Equal weights
weights = np.array([0.2, 0.2, 0.2, 0.2, 0.2])

# Portfolio value
portfolio_value = 1_000_000

print(f"Portfolio allocation:")
for ticker, weight in zip(tickers, weights):
    print(f"  {ticker}: {weight:.1%} (${portfolio_value * weight:,.0f})")

## 3. Calculate VaR Using Different Methods

In [ ]:
# Initialize VaR calculator (95% confidence)
var_calc = VaRCalculator(confidence_level=0.95)

# Calculate VaR using all methods
var_results = var_calc.calculate_all(
    returns=returns,
    portfolio_weights=weights,
    portfolio_value=portfolio_value
)

print("Value at Risk (95% Confidence):")
print("=" * 50)
for method, value in var_results.items():
    print(f"{method:20s}: ${value:,.2f}")

## 4. Visualize VaR Comparison

In [ ]:
fig = RiskVisualizer.plot_var_comparison(var_results, "VaR Comparison (95% Confidence)")
fig.show()

## 5. Calculate Component VaR

In [ ]:
# Calculate risk contribution of each asset
component_var = var_calc.var_breakdown(returns, weights)

# Scale by portfolio value
component_var_dollars = component_var * portfolio_value

print("Component VaR (Risk Contribution by Asset):")
print("=" * 50)
for asset, cvar in component_var_dollars.items():
    print(f"{asset:10s}: ${cvar:,.2f}")

# Visualize
fig = RiskVisualizer.plot_component_var(component_var_dollars)
fig.show()

## 6. Returns Distribution with VaR Overlay

In [ ]:
# Calculate portfolio returns
portfolio_returns = (returns * weights).sum(axis=1)

# Visualize distribution
fig = RiskVisualizer.plot_returns_distribution(
    portfolio_returns,
    var_value=var_results['historical_var'] / portfolio_value
)
fig.show()

## 7. Different Confidence Levels

In [ ]:
# Calculate VaR at different confidence levels
confidence_levels = [0.90, 0.95, 0.99]
var_by_confidence = {}

for cl in confidence_levels:
    calc = VaRCalculator(confidence_level=cl)
    var = calc.historical_var(returns, weights) * portfolio_value
    var_by_confidence[f"{cl:.0%}"] = var

print("Historical VaR at Different Confidence Levels:")
print("=" * 50)
for level, value in var_by_confidence.items():
    print(f"{level:10s}: ${value:,.2f}")